# CoordAnalyst - an educational tool to analyse coordination complexes


Claire Bachmann, Bertille Delloye, Enea Drezet--Marcot, Victor Davril


## Introduction

Coordination chemistry is an active field of research, with its study sitting at the intersection of many different branches of science, and its impact being major. In medicine, coordination compounds are indispensable. Cisplatin, one of the most widely used anticancer drugs in the world, is a simple platinum complex discovered in the 1960s that remains a frontline treatment for testicular, ovarian, and lung cancers. Another exmaple is Hemoglobin, the protein that carries oxygen through our blood, an iron coordination complex. In catalysis and industry, coordination chemistry drives some of the most economically significant chemical processes on the planet.The production of ammonia, acetaldehyde or even plastics, using the Haber-Bosch process, the Wacker process, and Ziegler-Natta catalysts, relies on iron, palladium and aluminium catalysts. In materials science, coordination compounds called metal-organic frameworks (MOFs) have emerged as one of the most exciting research areas of the 21st century, with applications in gas storage, carbon capture, drug delivery, and chemical sensing. Their extraordinary surface areas and tunable pore geometries make them uniquely powerful materials. In biology, nearly a third of all known proteins contain a metal ion at their active site : these coordination complexes perform catalysis, electron transfer, or structural stabilisation. Understanding these metalloenzymes is central to drug design and our understanding of life itself.  
Despite this enormous importance, coordination compounds remain challenging to characterise. Infrared and Raman spectroscopy are among the primary experimental tools used to study them, in order to reveal coordination modes, identify ligands, confirm geometries, and monitor reactions in real time. Yet interpreting these spectra requires deep chemical knowledge and access to reference data that is scattered across literature.  
Coordination chemistry is taught as a Bachelor's course in chemistry as the concepts are extremely relevant, and sometimes intertwine with spectroscopy and organometallic chemistry. Students are expected to aquire an understanding of metal and ligands, oxidation states, geometry, d-electrons, and bond stretches.    
Our package provides an educational tool with which students can try out different complexes' names and formulas and verify their properties, as well as visualize them in two and three dimensions. Moreover, CoordAnalyst provides an approximate IR and Raman spectra of the complex based on reference data from Nakamoto tables. It is presented as a unified interface, designed to make the spectral characterisation of coordination compounds more accessible to students and academics.



## Material and methods

### 1.1 Software architecture
The data is in a package outside of the main src code. In src, coordchem package : we have the files with general functions such as parsing the properties of the complex from the name and the formula, both functions are used in the file complex which creates an Object of class Complex. The geometry.py file allows to get the number of d-electrons on the metal and predict the geometry of the complex. 
Two packages were set up in coordchem : spectra and viz. Viz contains all the functions necessary to display the 2D and 3D visualisations of the molecule, such as the SMILES of the ligands and the index of the donor atom in ligand_data ; diagram_2d contains many functions necessary for the visualisation, transform_2d contains many mathematical functions to perform operation and make polydentate ligands visualisable, layout_2d gives the coordinates for the atoms in the different geometries. structure_3d gives a list of arrays + ?
The spectra package contains the predictor file which gives a PredictionResult object with all the spectrum bands extracted from the database in order of increasing wavenumber for an input ParsedComplex. The file renderer defines a gaussian and uses it to build the spectrum as a sum of gaussian peaks. It then defines the characteristics of the plot. 

### 1.2 Formula and Name Parser, Properties' extraction
The parsed information is stored in a central `Complex` object defined in **complex.py**. This unites the ParsedComplex object from formula and from name of the complex.  
This object contains the metal centre, the ligands, the ligand names, charges, denticity and donor atoms, the charge of the complex and total charge of the ligands, the counter ions and the raw formula, the oxidation state, the coordination number, as well as the predicted geometry and number of d-electrons using functions imported from **geometry.py**. This representation is important because it creates a common structure that can then be used by the geometry and visualisation modules.  
  
In **parser.py**, we first have a dictionnary used a database for the known ligands, counter ions, metals. The class `ParsedComplex`is defined here with the attributes mentionned above expect for the geometry and d-electrons. An object of the class is created with the parse_formula (main function) which calls on different functions to extract the properties of the complex.

In **name.py**, another dictionnary associates the symbol of the metals with their names (2 form possibles according to the charge of the complex), prefixes and roman numbers to numeric numbers. The name of the complex is turned into a formula with the functions metal_data, extract_complex_charge_from_name and ligand_data, so we can give the result as a ParsedComplex object imported from parser.py.

### 1.3 Spectral Band Database
The data was extracted systematically from  Nakamoto, K. "Infrared and Raman Spectra of Inorganic and Coordination Compounds", 6th ed., Wiley, 2009, and placed in SEED_BANDS : list of tuples with (ligand, coordination, metal, spectrum_type, wn_min, wn_max, intensity, assignment, ir_active, raman_active, source), which is defined at the top of the file as a BandRecord object. The data is given grouped by ligand for easy verification, and for both IR and Raman stretches.

Then the class IRBandDB is defined :  
The __init__ method does three things in order every time:  
-Opens a connection to SQLite  
-Creates the ir_bands table if it doesn't exist yet (_create_table)  
-Seeds the table with data if it's empty (_seed)  
   
First, _create_table runs a CREATE TABLE IF NOT EXISTS SQL statement. It defines the columns: ligand, coordination mode, metal, spectrum type, wavenumber range, intensity, assignment, whether it's IR/Raman active, and the source reference. Also creates two indexes to make lookups faster.
Then, _seed inserts all the rows from the SEED_BANDS list at the top of the file. It uses executemany which inserts all rows in one operation rather than one by one. The row count is checked first so it never double-inserts.  

The main query method is get_bands.   
(a) builds a SQL query with WHERE ligand = ? and AND spectrum_type = ?. The ? placeholders are essential to they prevent SQL injection and handle special characters safely.  
(b) if a coordination filter like "terminal" is passed, it adds AND coordination = ? to the query _row_to_record which return a list of BandRecord.  
(c) if a specific metal is passed : It splits the results into two groups: rows specific to that metal (e.g. metal = "Fe") and generic rows (metal = "any"). It then keeps all the specific rows, and only adds generic rows whose assignment isn't already covered by a specific row. This is how, for example, the Fe-specific 2093 cm⁻¹ band takes priority over the generic 2100–2200 range for the same C≡N stretch.  
get_all_ligands — returns a simple list of all ligand symbols in the database, e.g. ["CN", "CO", "Cl", "NH3", ...]. Useful for checking what's covered.  
get_bands_in_range(wmin, wmax) is used for reverse lookup : allows to identify an unknown peak in an experimental spectrum.  
add_band(...) is for inserting a custom entry. Useful when you encounter a ligand not in the seed data. Takes all the same fields as the database columns and commits immediately.  
summary(ligand) loops over IR then Raman bands and prints them as a formatted table, used for debugging and demos.  

### 1.4 Spectrum Prediction - Gaussians and plotting
**predictor.py** has a class PredictionResult which contains the attributes bands, intensities, the type of spectrum, the metal, the complex formula, ligand_coverage and warnings.  
The main function is predict_spectrum which returns a PredictionResult object from a ParsedComplex object. It loops over very ligand in the complex using the query db.get_bands for the specific ligand, spectrum and metal.   
   
Chemical corrections are applied :
- Backbonding shifts for π-accepting ligands (CN, CO, NO) with all metals: These ligands accept electron density from the metal via π-backbonding.
For more electron-rich metals, in low oxidation state, the backbonding is more important : weaker C≡N / C≡O bond leads to a lower wavenumber (red shift).
Similarly, less electron-rich metals or in high oxidation state allow for less backbonding : stronger bond in the ligand so the wavenumber is higher (blue shift).
- Coordination shifts : accounts for the shift in wavenumbers when the ligand binds to the metal center, compared to the wavenumber of the "free" ligand  contained in the database.
- Selection rules : we follow the mutual exclusion rule: In centrosymmetric complexes (of point group Oh, D4h), a band cannot be both IR and Raman active.

In predict_spectrum, we call the functions apply_backbonding_correction, apply_coordination_shift and apply_selection rules to create a list all_bands of CorrectedBand objects by modifying the bands from the database and adding them into the list.  

The band intensity is scaled, and not yet normalised, according to the number of ligand of the same type. The bands are sorted by order of increasing wavenumber to be returned. 

**renderer.py** first defines the mathematical gaussian for variables x, center, height and sigma. In build_spectrum : first we use linspace for x to create an array of evenly spaced numbers between wn_range[0] and wn_range[1]. We have decided 3000 points as the more points we have, the smoother the curve appears. For y, originally we create an array of the same size containing only zero. Then, looping over every band, a gaussian peak is added to the right y position.  
plot_spectrum uses x, y from build_spectrum and traces the figure. A vertical dotted line is also added from the top of the peak to the position on the x-axis to be able to easily read off the wavenumber of the stretch. 

### 1.5 2D and 3D visualizations 
The two-dimensional visualisation module generates schematic diagrams of coordination complexes. The central metal is placed at the centre of the figure, while the ligands are arranged around it according to the predicted coordination geometry. The aim is not to reproduce an exact crystallographic structure, but to provide a clear and chemically meaningful representation of the coordination sphere.

The layout is based on simple coordination chemistry rules. The coordination number defines an ideal reference geometry: linear for coordination number 2, trigonal planar for coordination number 3, tetrahedral or square planar for coordination number 4, octahedral for coordination number 6, and dodecahedral or square antiprismatic for coordination number 8. However, real coordination complexes are rarely perfectly ideal. Their structures can depend on metal size, d-electron count, ligand steric effects and electronic interactions. For example, four-coordinate complexes can be either tetrahedral or square planar: tetrahedral geometries minimise ligand--ligand repulsion, whereas square planar geometries are often favoured for low-spin \(d^8\) metal centres such as Pt(II), Pd(II) or Ni(II) with strong-field ligands. Six-coordinate complexes are commonly octahedral, but distortions such as tetragonal elongation, tetragonal compression or trigonal distortion may occur, especially for non-equivalent ligands or Jahn-Teller-active ions such as Cu(II).

In our implementation, we chose a simplified idealised model rather than attempting to reproduce these distortions quantitatively. The visualisation module uses ligand information such as denticity and ligand type to decide how many coordination sites each ligand occupies. Monodentate ligands occupy one coordination site, whereas chelating ligands occupy two neighbouring sites on the same metal centre. The layout was mainly designed to avoid overlap between ligands and to keep the diagram readable.

For octahedral complexes, the drawing follows a simple convention: one vertical bond is placed above the metal and one below it, while four equatorial bonds are arranged around the centre. These equatorial bonds are regularly separated in the 2D projection. Normal lines, bold wedges and dashed bonds are used to suggest the approximate three-dimensional orientation of the bonds. This convention makes the diagrams easier to interpret while remaining simple enough for an educational tool.

The current 2D module does not explicitly distinguish stereochemical isomers such as cis/trans, mer/fac or \(\Lambda/\Delta\). Instead, it provides a general idealised representation of the coordination environment. Overall, the module combines coordination number, ligand denticity and simplified bond styling to produce clear diagrams that can be used for teaching, comparison between complexes and inclusion in reports.

functions ?  
add 3D   

### 1.6 Web Application - Streamlit interface

victor

## Results

validation of geometry ? visualisations issues

### Database coverage
 IR spectroscopy is widely used to characterize coordination compounds, especially for identifying ligands such as CO, CN⁻, and NO, and distinguishing binding modes (e.g., SCN vs NCS). However, accurate spectral prediction typically requires quantum chemical calculations, which are computationally intensive and not always accessible. There is therefore value in a lightweight, interpretable tool that can provide rapid, approximate spectral predictions and assist with spectral assignment. The focus is on interpretability and rapid qualitative analysis rather than high-precision prediction.
The scope is restricted to a selected set of common ligands, including CO, CN⁻, NO, SCN⁻/NCS⁻, NO₂⁻/ONO⁻, NH₃, H₂O, halides, and several chelating ligands. (CN, CO, NH3, H2O, Cl, Br, I, OH, NO2, ONO, SCN, NCS, NO, N3, en, ox, acac, py, dmso, PPh3. ) The tool focuses on characteristic group frequencies rather than full vibrational analysis. 


## Discussion


### 3.1 Accuracy and limitations

restricted to one-metal-center only complexes, no bridging (?)

### 3.2 Value and comparison with existing tools

### 3.3 Future Improvements

# Conclusion